# Pseudobulk projection: CRC_Pelka2021

**Dataset:** Colorectal cancer tumor — Pelka et al. 2021  
💡 **Environment:** `clamp-analyses`

## Setup

In [ ]:
DATASET  = "CRC_Pelka2021"
OUT_ROOT = "output/03_model_biology/00_archs4/04_pseudobulk"
NORM_DIR = "output/01_model_building/05_pseudobulk"

## Load ARCHS4 CLAMP Hallmark model

In [ ]:
library(here)
model <- readRDS(here('output', '01_model_building', '04_archs4', '06_bp_coverage_rshall', '06_bp_coverage_hall_rs_100', 'hall_coverage_rs100_seed_1', 'CLAMPfull_hall.rds'))
cat('Model genes:', nrow(model$Z), '\n')
cat('Model LVs:  ', ncol(model$Z), '\n')

## Load preprocessed data

Reads `norm.csv` written by `00_preprocess.ipynb` (via Snakemake preprocess rule).

In [ ]:
# ---- Libraries ----
library(data.table)
library(dplyr)
library(CLAMP)
library(here)

OUTPUT_DIR <- file.path(here(), OUT_ROOT, DATASET)
dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)
cat("Output:", OUTPUT_DIR, "\n")

# ---- Load preprocessed norm.csv ----
norm_dt    <- fread(file.path(here(), NORM_DIR, DATASET, "norm.csv"))
norm_genes <- norm_dt[[1]]
norm       <- as.matrix(norm_dt[, -1, with = FALSE])
storage.mode(norm) <- "numeric"
rownames(norm) <- norm_genes
cat("Norm:", nrow(norm), "genes x", ncol(norm), "samples\n")

## Project into ARCHS4 model

In [ ]:
common <- intersect(rownames(model$Z), rownames(norm))
cat('Common genes:', length(common),
    sprintf('(%.1f%% of model)\n', 100 * length(common) / nrow(model$Z)))

m_sub   <- model
m_sub$Z <- as.matrix(model$Z[common, , drop = FALSE])

B <- CLAMP::projectCLAMP(CLAMPres = m_sub, newdata = norm[common, , drop = FALSE])
cat('Projection:', nrow(B), 'LVs x', ncol(B), 'samples\n')

## Save outputs

In [ ]:
write.csv(as.data.frame(B),
          file.path(OUTPUT_DIR, 'projection.csv'), row.names = TRUE)

# Export model summary for Python LV importance notebook
write.table(model$summary,
            file.path(OUTPUT_DIR, 'model_summary.tsv'),
            sep = '\t', row.names = FALSE, quote = FALSE)

message('Saved projection.csv and model_summary.tsv')

In [ ]:
proj_check <- read.csv(file.path(OUTPUT_DIR, 'projection.csv'), row.names = 1, check.names = FALSE)
cat('Saved projection dims:', nrow(proj_check), 'LVs x', ncol(proj_check), 'samples\n')